# part2.ipynb — Bias audit

**Prerequisite:** Run `part1.ipynb` first.


### i220524 FAST-NUCES Assignment 2 — Responsible & Explainable AI


## Environment setup

Install dependencies (local or Colab), then continue. On Colab: **Runtime → Change runtime type → GPU**.



In [ ]:
"""Install dependencies via `pip install -r requirements.txt` in a fresh environment (e.g. Colab)."""


def ensure_pkg():
    """No-op placeholder; run pip manually or automate here if desired."""
    return None


ensure_pkg()


In [ ]:
"""Imports and global configuration for reproducibility.

Sets USE_TF=0 before HuggingFace loads, and forces trainer_utils.is_tf_available() to False so Trainer.setup never imports TensorFlow (broken TF wheels still report as installed).
"""
import os
os.environ["USE_TF"] = "0"
import re
import random
import zipfile
from typing import Any, Dict, List, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.base import clone

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
import transformers.trainer_utils as _hf_trainer_utils
_hf_trainer_utils.is_tf_available = lambda: False
from datasets import Dataset

from fairlearn.postprocessing import ThresholdOptimizer
from fairlearn.metrics import demographic_parity_difference, equalized_odds_difference

from aif360.datasets import BinaryLabelDataset
from aif360.algorithms.preprocessing import Reweighing
from aif360.metrics import ClassificationMetric

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

ZIP_PATH = os.path.join(os.getcwd(), "jigsaw-unintended-bias-in-toxicity-classification.zip")
TRAIN_CSV_NAME = "train.csv"
ARTIFACT_DIR = os.path.join(os.getcwd(), "artifacts")
os.makedirs(ARTIFACT_DIR, exist_ok=True)

BASELINE_DIR = os.path.join(ARTIFACT_DIR, "baseline_distilbert")
POISON_DIR = os.path.join(ARTIFACT_DIR, "poisoned_distilbert")
REWEIGHT_DIR = os.path.join(ARTIFACT_DIR, "reweighted_distilbert")
OVERSAMPLE_DIR = os.path.join(ARTIFACT_DIR, "oversample_distilbert")
ISOTONIC_PATH = os.path.join(ARTIFACT_DIR, "isotonic_calibrator.joblib")

MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
NUM_EPOCHS = 3
BATCH_SIZE = 16
LR = 2e-5

"""Optional: set True only for quick smoke tests (not for submission)."""
FAST_DEBUG = False
if FAST_DEBUG:
    NUM_EPOCHS = 1


## Load data from the zip (stratified 100k train / 20k eval)

We read only the columns needed to limit memory. **Label:** `y = (target >= 0.5).astype(int)`. `train_test_split` returns `(larger_slice, test_size_slice)`; the 20k holdout is the **second** return value.



In [ ]:
"""Load `train.csv` from the competition zip without extracting the full archive to disk."""


def read_train_from_zip(zip_path: str, csv_name: str) -> pd.DataFrame:
    """Read selected columns for memory efficiency and normalize names."""
    usecols = [
        "comment_text",
        "target",
        "black",
        "white",
        "muslim",
        "jewish",
        "homosexual_gay_or_lesbian",
    ]
    with zipfile.ZipFile(zip_path, "r") as zf:
        with zf.open(csv_name) as f:
            df = pd.read_csv(f, usecols=usecols)
    df = df.rename(columns={"target": "toxic"})
    df["comment_text"] = df["comment_text"].fillna("").astype(str)
    for c in ["toxic", "black", "white", "muslim", "jewish", "homosexual_gay_or_lesbian"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)
    df["y"] = (df["toxic"] >= 0.5).astype(int)
    return df


df_all = read_train_from_zip(ZIP_PATH, TRAIN_CSV_NAME)
if FAST_DEBUG:
    eval_n = 2000
    train_n = 5000
    cap = eval_n + train_n + 5000
    df_all, _ = train_test_split(
        df_all, train_size=min(cap, len(df_all)), stratify=df_all["y"], random_state=SEED
    )
else:
    eval_n = 20000
    train_n = 100000

y = df_all["y"].values
train_remaining, eval_df = train_test_split(
    df_all, test_size=eval_n, stratify=y, random_state=SEED
)
y_rem = train_remaining["y"].values
train_df, _discard = train_test_split(
    train_remaining, train_size=train_n, stratify=y_rem, random_state=SEED
)

print("train_df:", train_df.shape, "eval_df:", eval_df.shape)
print("toxic rate train:", train_df["y"].mean(), "eval:", eval_df["y"].mean())


In [ ]:
"""Reload Part 1 checkpoint (run part1.ipynb first to create artifacts/baseline_distilbert)."""
tokenizer = AutoTokenizer.from_pretrained(BASELINE_DIR)
model = AutoModelForSequenceClassification.from_pretrained(BASELINE_DIR)
model.eval()
if torch.cuda.is_available():
    device = torch.device("cuda")
elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
model.to(device)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


def hf_predict_proba_toxic(texts, m, tok, device_obj, max_len):
    """Return toxic-class probabilities for a list of strings."""
    enc = tok(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device_obj)
    with torch.no_grad():
        logits = m(**enc).logits
        pr = torch.softmax(logits, dim=-1)[:, 1].detach().cpu().numpy()
    return pr


texts_eval = eval_df["comment_text"].tolist()
y_true = eval_df["y"].to_numpy()
scores = hf_predict_proba_toxic(texts_eval, model, tokenizer, device, MAX_LENGTH)
CHOSEN_THRESHOLD = 0.5
thr = CHOSEN_THRESHOLD
y_hat = (scores >= thr).astype(int)


# Part 2 — Bias audit (high-black vs reference cohort)

**Cohorts (evaluation subset):**
- **High-black:** `black >= 0.5`
- **Reference:** `black < 0.1` and `white >= 0.5`



In [ ]:
"""Bias audit metrics, disparate impact ratio, aif360 metrics, plots."""
cohort_high = eval_df["black"] >= 0.5
cohort_ref = (eval_df["black"] < 0.1) & (eval_df["white"] >= 0.5)

print("high-black cohort:", int(cohort_high.sum()))
print("reference cohort:", int(cohort_ref.sum()))


def cohort_metrics(y_true_arr: np.ndarray, y_score: np.ndarray, thr: float) -> Dict[str, float]:
    """TPR on toxic, FPR on non-toxic, FNR on toxic, precision on positives for a threshold."""
    y_hat = (y_score >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true_arr, y_hat).ravel()
    tpr_toxic = tp / (tp + fn) if (tp + fn) > 0 else float("nan")
    fpr_nontox = fp / (fp + tn) if (fp + tn) > 0 else float("nan")
    fnr_toxic = fn / (tp + fn) if (tp + fn) > 0 else float("nan")
    prec_flag = tp / (tp + fp) if (tp + fp) > 0 else float("nan")
    return {
        "TPR_toxic": tpr_toxic,
        "FPR_nontoxic": fpr_nontox,
        "FNR_toxic": fnr_toxic,
        "Precision_flagged": prec_flag,
    }


def build_aif360_dataset(labels: np.ndarray, sens: np.ndarray) -> BinaryLabelDataset:
    """Build aif360 dataset; label=1 means toxic; race is the sensitive attribute."""
    df_b = pd.DataFrame({"label": labels.astype(float), "race": sens.astype(float)})
    return BinaryLabelDataset(
        favorable_label=0.0,
        unfavorable_label=1.0,
        df=df_b,
        label_names=["label"],
        protected_attribute_names=["race"],
        privileged_protected_attributes=[[0.0]],
    )


thr = CHOSEN_THRESHOLD
scores_eval = scores
y_hat = (scores_eval >= thr).astype(int)

m_high = cohort_metrics(y_true[cohort_high.values], scores_eval[cohort_high.values], thr)
m_ref = cohort_metrics(y_true[cohort_ref.values], scores_eval[cohort_ref.values], thr)

di = m_high["FPR_nontoxic"] / m_ref["FPR_nontoxic"] if m_ref["FPR_nontoxic"] > 0 else float("nan")

summary = pd.DataFrame(
    [
        {"cohort": "high_black", **m_high},
        {"cohort": "reference", **m_ref},
    ]
)
print(summary)
print("Disparate impact ratio (FPR_high / FPR_ref):", di)

sens = np.where(cohort_high, 1.0, 0.0)
sens[~cohort_high & ~cohort_ref] = 0.0

ds_true = build_aif360_dataset(y_true, sens)
ds_pred = build_aif360_dataset(y_hat, sens)

cm_obj = ClassificationMetric(
    ds_true,
    ds_pred,
    unprivileged_groups=[{"race": 1.0}],
    privileged_groups=[{"race": 0.0}],
)
print("aif360 statistical parity difference:", cm_obj.statistical_parity_difference())
print("aif360 equal opportunity difference:", cm_obj.equal_opportunity_difference())

labels_bar = ["TPR", "FPR", "FNR"]
x = np.arange(len(labels_bar))
w = 0.35
vals_high = [m_high["TPR_toxic"], m_high["FPR_nontoxic"], m_high["FNR_toxic"]]
vals_ref = [m_ref["TPR_toxic"], m_ref["FPR_nontoxic"], m_ref["FNR_toxic"]]

plt.figure(figsize=(8, 5))
plt.bar(x - w / 2, vals_high, width=w, label="high-black")
plt.bar(x + w / 2, vals_ref, width=w, label="reference")
plt.xticks(x, labels_bar)
plt.ylabel("rate")
plt.title("Group error rates (chosen threshold)")
plt.legend()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))


def plot_cm(ax, yt, yp, title):
    """Plot a confusion matrix heatmap."""
    cm_local = confusion_matrix(yt, yp)
    sns.heatmap(cm_local, annot=True, fmt="d", cmap="Blues", ax=ax)
    ax.set_xlabel("predicted")
    ax.set_ylabel("true")
    ax.set_title(title)


plot_cm(axes[0], y_true[cohort_high.values], y_hat[cohort_high.values], "high-black cohort")
plot_cm(axes[1], y_true[cohort_ref.values], y_hat[cohort_ref.values], "reference cohort")
plt.tight_layout()
plt.show()


### Part 2 — Interpretation (model answer — tighten to your printed metrics)

**Largest disparity:** Often **FPR on non-toxic comments** differs most between cohorts (check **disparate impact ratio** and **SPD**). **EOD** shows whether **TPR** is skewed too, or if the main issue is **who gets falsely flagged**.

**Direction:** **Higher FPR** on the high-black cohort ⇒ **over-flagging** (innocent comments penalized). **Higher FNR** there ⇒ **under-flagging** (toxic content left up). **Both** can show up; your bar chart and confusion matrices show which dominates.

**Harm / risk:** Over-flagging chills speech and erodes trust where errors cluster; under-flagging leaves abuse targets less protected. For the platform, over-flagging raises **fairness and bias** exposure; under-flagging raises **safety and reputation** exposure.

**Edit for submission:** Replace “often” with your **biggest gap** (e.g. FPR ratio, SPD, or EOD) in **one sentence** tied to your outputs.

